## Cohort Construction

Constructs the final modelling cohort for AKI prediction by applying the inclusion criteria

outputs: merged_icu.parquet, cohort.parquet

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns",None)


In [ ]:

PROJECT_DIR = Path.cwd()

DATA_PATH = PROJECT_DIR / "Mimic"

HOSP = DATA_PATH / "hosp"
ICU = DATA_PATH / "icu"

OUTPUT = PROJECT_DIR / "processed_outputs"
OUTPUT.mkdir(exist_ok=True)

In [ ]:
patients = pd.read_csv(HOSP / "patients.csv",usecols=["subject_id","gender","anchor_age","dod"])

In [ ]:
admissions = pd.read_csv(HOSP / "admissions.csv",usecols=["subject_id","hadm_id", "admission_type","admittime", "dischtime"])

In [ ]:
icustays = pd.read_csv(ICU / "icustays.csv",usecols=["subject_id", "hadm_id","stay_id","first_careunit","intime","outtime","los" ])

In [ ]:
admissions_date_columns = ["admittime", "dischtime"]

for col in admissions_date_columns:
    admissions[col] = pd.to_datetime(admissions[col])


In [ ]:

icustays_date_columns = ["intime","outtime"]

for col in icustays_date_columns:
    icustays[col] = pd.to_datetime(icustays[col] )

In [ ]:

print("Patients")

print(patients.shape)

print()

print("Admissions")

print(admissions.shape)

print()

print("ICU Stays")

print(icustays.shape)

In [ ]:
patient_admissions = admissions.merge(patients,on="subject_id",how="left",
    validate="many_to_one") # to make sure one admission belongs to one patient

In [ ]:
merged_icu = icustays.merge(patient_admissions,on=["subject_id", "hadm_id"],how="left",
    validate="many_to_one")

In [ ]:
print(merged_icu.shape)
merged_icu.head()

In [ ]:
merged_icu.isna().sum()

In [ ]:
all_icu_stays = merged_icu["stay_id"].nunique()
all_patients = merged_icu["subject_id"].nunique()
print(all_icu_stays)
print(all_patients)

In [ ]:
merged_icu.to_parquet(OUTPUT / "merged_icu.parquet",index=False)

In [ ]:
cohort = merged_icu[merged_icu["anchor_age"] >= 18].copy()

adult_icu_stays = cohort["stay_id"].nunique()
adult_patients = cohort["subject_id"].nunique()

In [ ]:
print(cohort.shape)
print(adult_icu_stays)
print(adult_patients)

In [ ]:
cohort = (cohort.sort_values("intime").drop_duplicates(subset="subject_id",keep="first"))

first_icu_stays = cohort["stay_id"].nunique()
first_icu_patients = cohort["subject_id"].nunique()
print(first_icu_stays)
print(first_icu_patients)

In [ ]:
cohort = cohort[cohort["los"] >= 1].copy()

final_icu_stays = cohort["stay_id"].nunique()
final_patients = cohort["subject_id"].nunique()
print(final_icu_stays)
print(final_patients)

In [ ]:
cohort.reset_index(drop=True,inplace=True)

In [ ]:
print("Patients")

print(cohort["subject_id"].nunique())

print("ICU Stays")

print(cohort["stay_id"].nunique())

In [ ]:
cohort["gender"].value_counts()

In [ ]:
cohort["anchor_age"].value_counts()

In [ ]:
cohort["first_careunit"].value_counts()

In [ ]:
cohort.to_parquet(OUTPUT / "cohort.parquet",index=False)

In [ ]:
print("Final Cohort Size")
print(cohort.shape)

In [ ]:
cohort_flow = pd.DataFrame({
    "Step": [
        "All ICU stays",
        "Adults (≥18 years)",
        "First ICU admission",
        "ICU LOS ≥24 hours"
    ],
    "ICU Stays": [
        all_icu_stays,
        adult_icu_stays,
        first_icu_stays,
        final_icu_stays
    ],
    "Unique Patients": [
        all_patients,
        adult_patients,
        first_icu_patients,
        final_patients
    ]
})

In [ ]:
print(cohort_flow)

In [ ]:
cohort_flow.to_csv(OUTPUT / "cohort_flow.csv",index=False)